In [1]:
import os
import glob
import pandas as pd
import numpy as np
import datetime
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from itertools import product

In [3]:
import math

In [4]:
from scipy.signal import butter, filtfilt

In [5]:
import kneed
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler

# Open Sound Files

In [6]:
sound_suvarnphumi = '../sound_proj_2/data/Raw-data/VTBS-SuvarnphumiAirport/2016/Leq0.5sec-VTBS-Date2MARto6MAR2016 ...xlsx'
sound_maepahluang = '../sound_proj_2/data/Raw-data/MahPahLuangAirport/Leq0.5s-VTCT-Date20to25Mar2019.xlsx'

flight_suvarnphumi = '../sound_proj_2/data/Raw-data/VTBS-SuvarnphumiAirport/2016/Leq0.5sec-VTBS-Date2MARto6MAR2016 ...xlsx'
flight_maepahluang = '../sound_proj_2/data/Raw-data/MahPahLuangAirport/Billing-ท่าอากาศยานแม่ฟ้าหลวง เชียงราย.xlsx'

In [7]:
def read_excel_file(excelfile):
    xl = pd.ExcelFile(excelfile)
    sheet_list = xl.sheet_names

    for c in ['Billing', 'billing']:
          if c in sheet_list:
               sheet_list.remove(c)
    
    sound_files = [pd.read_excel(excelfile, header=None, sheet_name=sn) for sn in sheet_list]
    return sound_files

In [8]:
def parse_date(dt):
	date_format = ['%m/%d/%Y %I:%M:%S %p:%f', '%Y/%m/%d %H:%M:%S', '%m/%d/%Y %H:%M:%S', '%Y/%m/%d %H:%M:%S',
	        	   '%Y-%m-%d', '%m-%d-%Y', '%d-%m-%Y', '%Y/%m/%d', '%m/%d/%Y', '%d/%m/%Y',
				   '%Y-%m-%d %H:%M:%S', '%m-%d-%Y %H:%M:%S', '%d-%m-%Y %H:%M:%S',
				   '%Y/%m/%d %H:%M:%S', '%m/%d/%Y %H:%M:%S', '%d/%m/%Y %H:%M:%S']

	for date_f in date_format:
		try:
			dateObject = datetime.datetime.strptime(dt, date_f)
			break
		except:
			dateObject = None
	
	return dateObject

In [9]:
def calculate_pts(df):
    one_hr_pts = 36000
    pts_period = ((df.iloc[1]['Period start'] - df.iloc[0]['Period start']) / np.timedelta64(1, 's')) * 10
    one_hr_result = int(one_hr_pts / pts_period)
    return one_hr_result

In [10]:
def data_prepare(sound_files):
    config_files = [dict(zip(sf[0:8][0], sf[0:8][1])) for sf in sound_files]

    sound_files = [pd.concat([sf,pd.DataFrame({0:[config_files[i]['End']],1:[np.nan]})], ignore_index = True) for i, sf in enumerate(sound_files)]
    data_list = [pd.DataFrame(sf[9:].values, columns=sf[8:9].values[0]) for sf in sound_files]

    df = pd.concat(data_list)
    df = df[~(df['Period start'].str.contains("Overall"))].reset_index(drop=True)

    df_clean = df.copy()
    df_clean['Period start'] = df_clean['Period start'].apply(lambda a: parse_date(a))
    df_clean.fillna(method='ffill', inplace=True)

    return df_clean

In [11]:
def cutoff(r, data):
    r = (100 - r) / 1000
    b, a = butter(4, r)
    filtered = filtfilt(b, a, data)
    return filtered

In [12]:
def standardize(data):
    sound = np.array(data)
    sound = sound.reshape(-1,1)

    scaler = StandardScaler()
    sound_normalized = scaler.fit_transform(sound)

    return sound_normalized

In [13]:
def sound_cluster(data):
    sse = {k : KMeans(n_clusters=k, random_state=0).fit(data).inertia_ for k in range(1, 5)}

    kn = kneed.KneeLocator(
        x=list(sse.keys()), 
        y=list(sse.values()), 
        curve='convex', 
        direction='decreasing', S=0.0)
    
    k_best = kn.knee
    kmeans_best = KMeans(n_clusters=k_best, random_state=0).fit(data)

    return kmeans_best

In [14]:
def find_peaks(data, time):
    data_copy = data.copy()
    X = time.reshape(-1, 1)
    dbscan = DBSCAN(eps=150, min_samples=10).fit(X)

    data_copy = data_copy.iloc[time]
    data_copy['peak_group'] = dbscan.labels_
    data_copy = data_copy[data_copy['peak_group'] != -1]
    return data_copy['peak_group']

In [15]:
def merge_data(df1, df2):
    df_merge = pd.concat([df1, df2], axis=1)
    df_merge['peak_group'] = df_merge['peak_group'].fillna(-1)
    return df_merge

In [16]:
def computeL_eq_t(data):
    sum_pow = np.sum(np.power(10, data/10))
    x = 1/len(data) * sum_pow
    return 10 * math.log(x,10)

def SoundAddition(data):
    x = np.sum(np.power(10, data/10))
    return 10 * math.log(x,10)

def computeSEL(data):
    return computeL_eq_t(data) + 10 * math.log((len(data)*100)/1000)

In [17]:
def IQR_outlier(data, d_type):
    Q1 = np.percentile(data[d_type], 25)
    Q3 = np.percentile(data[d_type], 75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    df_result_cleaned = data[~((data[d_type] < lower_bound) | (data[d_type] > upper_bound))]
    return df_result_cleaned

In [18]:
def find_localpeak_alternative(d, data):
    df_copy = data.copy()
    one_hr_result = calculate_pts(df_copy)

    days_dict = {"half day":one_hr_result * 12, "day":one_hr_result * 24, "week":one_hr_result * 24 * 7}
    df_copy = df_copy[:days_dict[d]+1]

    sound_cutoff = cutoff(70, df_copy['Leq'])
    df_copy = df_copy.assign(Leq_filtered=sound_cutoff)

    df_copy['Leq_norm'] = standardize(df_copy['Leq_filtered'])
    sound_norm = np.array(df_copy['Leq_norm']).reshape(-1,1)

    x_kmeans = sound_cluster(sound_norm)
    df_copy['cluster'] = x_kmeans.labels_
    
    avg_cluster = [df_copy[df_copy['cluster'] == i]['Leq_filtered'].mean() for i in range(len(np.unique(x_kmeans.labels_)))]
    max_cluster = np.array(avg_cluster).argmax()
    time_on_max_clusters = np.array(df_copy[df_copy['cluster'] == max_cluster].index)
    
    peaks = find_peaks(df_copy, time_on_max_clusters)
    df_result = merge_data(df_copy, peaks)

    peak_median = [np.median(df_result[df_result['peak_group'] == i].index) for i in peaks.unique() if (len(df_result[df_result['peak_group'] == i]) != 0)]
    unix_time = np.array(df_result['Period start'])
    results = []

    for c in peaks.unique():
        c_data = df_result[df_result['peak_group'] == c].index
        min_t, max_t = c_data.min(), c_data.max()
        peak_t = int(peak_median[c])

        in_which_hour = min_t//36000

        raw = np.array(df_result[df_result['peak_group'] == c]['Leq'])
        Leq = computeL_eq_t(raw)
        LAE = computeSEL(raw)

        r = {"hours":in_which_hour, "start_time": unix_time[min_t], "end_time": unix_time[max_t], "peak_time": peak_t,
             "interval": unix_time[max_t]-unix_time[min_t], "median":peak_median[c], "Leq":Leq, "Lae":LAE}

        results.append(r)

    df_result = pd.DataFrame(results)
    df_result['interval'] = df_result['interval'] / np.timedelta64(1, 's')

    df_result_cleaned = IQR_outlier(df_result, 'interval')

    return df_result_cleaned

In [19]:
def to_time(t):
    t_dt = pd.to_datetime(t)
    return t_dt.time()

def to_date(dt):
    try:
        dt = pd.to_datetime(dt)
        return dt
    except:
        dateObject = datetime.datetime.strptime(dt, '%Y-%m-%d %H:%M:%S')
        dt_str = dateObject.date()

        return datetime.date(dt_str.year-543, dt_str.month, dt_str.day)

In [29]:
def find_accuracy(lag_width, flights_file, peaks):
    def match_cond(start,stop,date_time):
        return (date_time >= start-datetime.timedelta(milliseconds=lag_width*1000)) & (date_time <= stop+datetime.timedelta(milliseconds=lag_width*1000))
    
    if "LOCAL DATE.1" in flights_file.columns:
        flights_file.rename(columns = {"LOCAL DATE.1":"LOCAL TIME"}, inplace = True)

    flights_file['LOCAL DATE'] = flights_file['LOCAL DATE'].astype('str')
    flights_file['LOCAL TIME'] = flights_file['LOCAL TIME'].astype('str')

    flights_file["LOCAL TIME"] = flights_file["LOCAL TIME"].apply(lambda a: to_time(a))
    flights_file["LOCAL DATE"] = flights_file["LOCAL DATE"].apply(lambda a: to_date(a))
    
    flights_file["Datetime"] = pd.to_datetime(flights_file["LOCAL DATE"].astype('str') + " " + flights_file["LOCAL TIME"].astype('str'))
    
    peaks["start_time"], peaks["end_time"] = pd.to_datetime(peaks["start_time"]), pd.to_datetime(peaks["end_time"])

    min_peak, max_peak = min(peaks["start_time"]), max(peaks["end_time"])
    
    flights_file = flights_file[(flights_file['Datetime'] > min_peak) & (flights_file['Datetime'] < max_peak)]
    
    matched = [flights_file[match_cond(start,stop,flights_file['Datetime'])] for start, stop in zip(peaks['start_time'], peaks['end_time'])
                if len(flights_file[match_cond(start,stop,flights_file['Datetime'])]) > 0]
    
    return len(matched) / len(peaks)

In [21]:
sound_suvarnphumi_transform = read_excel_file(sound_suvarnphumi)
sound_maepahluang_transform = read_excel_file(sound_maepahluang)

In [22]:
df_flights_suvarnphumi = pd.read_excel(flight_suvarnphumi, sheet_name='Billing')
df_flights_maepahluang = pd.read_excel(flight_maepahluang)

In [23]:
sound_dict = {'Suvarnphumi Airport':sound_suvarnphumi_transform, 'MahPahLuang Airport':sound_maepahluang_transform}
flight_dict = {'Suvarnphumi Airport':df_flights_suvarnphumi, 'MahPahLuang Airport':df_flights_maepahluang}

In [31]:
data_list = []

places = ['Suvarnphumi Airport', 'MahPahLuang Airport']
days = ['half day', 'day', 'week']
lag_width_list = [15,30,60]

for pl, d, lag_w in product(places, days, lag_width_list):
    df = data_prepare(sound_dict[pl])
    df_sound = find_localpeak_alternative(d, df)
    acc = find_accuracy(lag_w, flight_dict[pl], df_sound)
    data_list.append({'place':pl, 'days':d, 'lag width': lag_w, 'accuracy score':acc})
    print(f'{pl} days={d} lag_width={lag_w} finished...')

Suvarnphumi Airport days=half day lag_width=15 finished...
Suvarnphumi Airport days=half day lag_width=30 finished...
Suvarnphumi Airport days=half day lag_width=60 finished...
Suvarnphumi Airport days=day lag_width=15 finished...
Suvarnphumi Airport days=day lag_width=30 finished...
Suvarnphumi Airport days=day lag_width=60 finished...
Suvarnphumi Airport days=week lag_width=15 finished...
Suvarnphumi Airport days=week lag_width=30 finished...
Suvarnphumi Airport days=week lag_width=60 finished...
MahPahLuang Airport days=half day lag_width=15 finished...
MahPahLuang Airport days=half day lag_width=30 finished...
MahPahLuang Airport days=half day lag_width=60 finished...
MahPahLuang Airport days=day lag_width=15 finished...
MahPahLuang Airport days=day lag_width=30 finished...
MahPahLuang Airport days=day lag_width=60 finished...
MahPahLuang Airport days=week lag_width=15 finished...
MahPahLuang Airport days=week lag_width=30 finished...
MahPahLuang Airport days=week lag_width=60 fini

In [32]:
df = pd.DataFrame(data_list)
df.to_csv("../sound_proj_2/data/Processed-data/output_viz2.csv",index=False)

# Data Viz

In [35]:
import plotly.express as px

In [33]:
df_viz = pd.read_csv('../sound_proj_2/data/Processed-data/output_viz2.csv')
df_viz.head()

,place,days,lag width,accuracy score
0,Suvarnphumi Airport,half day,15,0.621053
1,Suvarnphumi Airport,half day,30,0.789474
2,Suvarnphumi Airport,half day,60,0.863158
3,Suvarnphumi Airport,day,15,0.647541
4,Suvarnphumi Airport,day,30,0.786885


In [34]:
df_viz_half_day = df_viz[df_viz['days'] == 'half day']
df_viz_day = df_viz[df_viz['days'] == 'day']
df_viz_week = df_viz[df_viz['days'] == 'week']

In [36]:
fig = px.histogram(df_viz_half_day, x="place", y="accuracy score", color='lag width', text_auto='.4f'
                   , barmode='group', height=400, title="Accuracy score of Aircraft Calculate in half day")
fig.show()

In [37]:
fig = px.histogram(df_viz_day, x="place", y="accuracy score", color='lag width', text_auto='.4f'
                   , barmode='group', height=400, title="Accuracy score of Aircraft Calculate in day")
fig.show()

In [38]:
fig = px.histogram(df_viz_week, x="place", y="accuracy score", color='lag width', text_auto='.4f'
                   , barmode='group', height=400, title="Accuracy score of Aircraft Calculate in week")
fig.show()